# QLoRA Fine-Tuning Tutorial - 8GB VRAM

This notebook demonstrates QLoRA fine-tuning optimized for 8GB VRAM GPUs.

## What You'll Learn
- Load a pre-trained model with 4-bit quantization
- Setup LoRA adapters
- Train on instruction dataset
- Export to Ollama format

## Requirements
- GPU with 8GB+ VRAM
- Python 3.9+
- 8GB+ RAM

In [ ]:
# Install dependencies
!pip install -q transformers peft datasets accelerate bitsandbytes trl

In [ ]:
# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB" if torch.cuda.is_available() else "No GPU")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Load dataset
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca", split="train").select(range(500))
print(f"Samples: {len(dataset)}")
print(dataset[0])

In [ ]:
# Format instructions
def format_example(example):
    if example.get("input") and example["input"].strip():
        text = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    else:
        text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    return {"text": text}

dataset = dataset.map(format_example)
print(dataset[0]["text"][:200])

In [ ]:
# Load model with 4-bit quantization
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Model loaded: {model_name}")
print(f"Parameters: {model.num_parameters()/1e6:.1f}M")

In [ ]:
# Setup LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Training
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=2,
    gradient_checkpointing=True,
    logging_steps=10,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=256,
)

print("Starting training...")
trainer.train()

In [ ]:
# Save adapter
trainer.save_model("./adapter")
tokenizer.save_pretrained("./adapter")
print("Adapter saved to ./adapter")

In [ ]:
# Test inference
import torch

model.eval()
prompt = "What is machine learning?"
messages = [{"role": "user", "content": prompt}]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)

print("\nPrompt:", prompt)
print("Response:", tokenizer.decode(outputs[0], skip_special_tokens=True))

## Next Steps

1. Merge adapter with base model
2. Convert to GGUF format
3. Register with Ollama

```bash
python scripts/merge.py --adapter ./adapter --output ./merged
python scripts/convert.py --model ./merged --output model.gguf
ollama create my-model -f Modelfile
```